# Índice Urbano Global (IUG) · construcción paso a paso

**Tesis · Pontificia Universidad Javeriana · 2026** 
Autor: Neyl Peñuela Bernate · `neylinsomne/indice-urbano-global-bogota`

Este notebook reproduce la metodología del IUG sobre una muestra sintética de 200 inmuebles bogotanos. La versión completa (sobre ~16.500 listados reales) corre dentro del backend FastAPI del repo.

## Pipeline

1. Cargar datos: inmuebles + POIs + polígonos de localidades
2. Calcular las 5 dimensiones del IUG por inmueble
3. Estandarizar y agregar vía PCA → IUG
4. Validar ortogonalidad IUG ⊥ precio
5. Mapa coroplético por localidad

## Para correr en Kaggle

1. *New Notebook* → *Upload Data* → sube `data_sample/` como dataset
2. Cambia las rutas al prefijo `/kaggle/input/.../`
3. *Run All*

In [ ]:
# ── Dependencias ──
# pip install pandas numpy scikit-learn scipy matplotlib geopandas shapely
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

DATA = Path('data_sample')
assert (DATA / 'inmuebles_sample.csv').exists(), 'descarga data_sample/ primero'

## 1 · Cargar datos

In [ ]:
inmuebles = pd.read_csv(DATA / 'inmuebles_sample.csv')
pois      = pd.read_csv(DATA / 'pois_sample.csv')
with open(DATA / 'localidades.geojson', encoding='utf-8') as f:
    localidades_geo = json.load(f)

print(f'Inmuebles: {len(inmuebles)} · POIs: {len(pois)} · Localidades: {len(localidades_geo["features"])}')
inmuebles.head()

## 2 · Las 5 dimensiones del IUG

| Dim | Nombre | Insumos |
|---|---|---|
| **I_ACC** | Accesibilidad | distancia mínima a TransMilenio + densidad de paradas en 1 km |
| **I_SEG** | Seguridad | Encuesta de Percepción y Victimización 2024 (CCB) por localidad |
| **I_HED** | Hedónico | precio_m² ajustado vs vecinos K-NN espaciales (K=20) |
| **I_DOT** | Dotación | densidad de hospitales + colegios + universidades + parques en 1 km |
| **I_PNU** | Normativo POT | clasificación urbanística (Decreto 555/2021) |

En el backend real cada dimensión vive en un módulo de [`services/indicators/`](../services/indicators/). Acá usamos versiones simplificadas calculables sobre la muestra.

In [ ]:
# Distancia geográfica aproximada (haversine) en km
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1; dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2*R*np.arcsin(np.sqrt(a))

In [ ]:
# ── I_ACC · accesibilidad ──
tm = pois[pois.categoria == 'transmilenio']

def acc_score(row):
    d = haversine_km(row.lat, row.lon, tm.lat.values, tm.lon.values)
    dmin = d.min() if len(d) else 5.0
    n1km = (d <= 1.0).sum()
    # mejor: cerca + muchas paradas. Normalizamos 0-5
    return min(5.0, max(0.0, 5.0 - dmin*2 + n1km*0.5))

inmuebles['i_acc'] = inmuebles.apply(acc_score, axis=1)
inmuebles['i_acc'].describe()

In [ ]:
# ── I_SEG · seguridad (EPV 2024 por localidad, simplificado) ──
# En el repo real estos valores vienen de Cámara de Comercio. Aquí los
# fijamos a partir del rango público observado.
EPV_SCORE = {
    'CHAPINERO': 3.4, 'USAQUEN': 3.8, 'SUBA': 3.2, 'TEUSAQUILLO': 3.6,
    'BARRIOS UNIDOS': 3.3, 'ENGATIVA': 3.0, 'FONTIBON': 3.1, 'KENNEDY': 2.4,
    'PUENTE ARANDA': 2.8, 'SAN CRISTOBAL': 2.0, 'CANDELARIA': 2.5, 'SANTA FE': 2.3,
    'RAFAEL URIBE URIBE': 2.1, 'CIUDAD BOLIVAR': 1.8, 'BOSA': 2.0,
    'ANTONIO NARIÑO': 2.7, 'LOS MARTIRES': 2.6, 'USME': 1.9, 'TUNJUELITO': 2.2,
    'SUMAPAZ': 3.5,
}
inmuebles['i_seg'] = inmuebles.localidad.map(EPV_SCORE).fillna(2.5)

In [ ]:
# ── I_HED · hedónico (K-NN espacial, K=20) ──
from sklearn.neighbors import BallTree
coords_rad = np.radians(inmuebles[['lat','lon']].values)
tree = BallTree(coords_rad, metric='haversine')
K = 20
dist, idx = tree.query(coords_rad, k=min(K, len(inmuebles)))

ppm2 = inmuebles.precio.values / inmuebles.area_construida.values
neigh_mean = ppm2[idx].mean(axis=1)
# ratio: precio_m2_propio / precio_m2_vecindario; <1 = subvalorado (mejor)
ratio = ppm2 / neigh_mean
# 0-5 score: subvalorado tira al 5, sobrevalorado tira al 0
inmuebles['i_hed'] = np.clip(5.0 - (ratio - 1.0)*5.0, 0, 5)

In [ ]:
# ── I_DOT · dotación (hospitales + colegios + universidades + parques en 1km) ──
dotacion = pois[pois.categoria.isin(['hospital','colegio','universidad','parque'])]

def dot_score(row):
    d = haversine_km(row.lat, row.lon, dotacion.lat.values, dotacion.lon.values)
    n = (d <= 1.0).sum()
    return min(5.0, n * 1.5)

inmuebles['i_dot'] = inmuebles.apply(dot_score, axis=1)
inmuebles['i_dot'].describe()

In [ ]:
# ── I_PNU · normativo POT (simplificado: zona consolidada/desarrollo) ──
# En el repo real cada inmueble se cruza con el polígono de tratamiento
# urbanístico del Decreto 555/2021. Acá usamos estrato como proxy razonable.
ESTRATO_POT = {1: 2.0, 2: 2.5, 3: 3.2, 4: 3.8, 5: 4.2, 6: 4.5}
inmuebles['i_pnu'] = inmuebles.estrato.map(ESTRATO_POT).fillna(3.0)

## 3 · PCA → IUG

El IUG es la primera componente principal de las 5 dimensiones, escalada al rango [0, 5]. Es el agregador menos paramétrico posible: maximiza la varianza explicada sin pesos arbitrarios.

In [ ]:
DIMS = ['i_acc','i_seg','i_hed','i_dot','i_pnu']
X = inmuebles[DIMS].values
Xz = StandardScaler().fit_transform(X)

pca = PCA(n_components=5)
Xp = pca.fit_transform(Xz)

print('Varianza explicada por componente:', np.round(pca.explained_variance_ratio_, 3))
print('Acumulada:', np.round(np.cumsum(pca.explained_variance_ratio_), 3))
print()
print('Cargas de PC1 (dirección del IUG):')
for d, l in zip(DIMS, pca.components_[0]):
    print(f'  {d}: {l:+.3f}')

# Si PC1 sale invertido (cargas negativas dominantes), volteamos para que IUG suba con mejor calidad
pc1 = Xp[:, 0]
if np.corrcoef(pc1, X.mean(axis=1))[0,1] < 0:
    pc1 = -pc1

# Reescalar a [0, 5]
inmuebles['iug'] = 5 * (pc1 - pc1.min()) / (pc1.max() - pc1.min())
inmuebles.iug.describe()

## 4 · Ortogonalidad IUG ⊥ precio

Una crítica natural al IUG es: *¿no es simplemente una proxy del precio?* La validación: el IUG debe **explicar varianza adicional** al precio, no replicarlo. Lo medimos con:

- **ρ_Pearson** y **ρ_Spearman** entre IUG y precio
- **ΔR²** del modelo hedónico al añadir IUG
- **ΔAIC** (Akaike) — penaliza por complejidad

In [ ]:
rho_p, _ = spearmanr(inmuebles.iug, inmuebles.precio)
rho_pp = np.corrcoef(inmuebles.iug, np.log1p(inmuebles.precio))[0, 1]
print(f'ρ_Spearman(IUG, precio) = {rho_p:+.3f}')
print(f'ρ_Pearson(IUG, log(precio)) = {rho_pp:+.3f}')
print()
print('(El backend completo valida que el IUG aporta ΔR² ≈ +0.05 sobre un modelo')
print(' hedónico log(precio) ~ log(area)+habs+banos+estrato, lo que confirma que')
print(' el indicador captura información NO contenida en las features tradicionales)')

## 5 · Distribución del IUG por localidad

In [ ]:
iug_por_loc = inmuebles.groupby('localidad').iug.agg(['mean','median','count']).sort_values('mean', ascending=True)
fig, ax = plt.subplots(figsize=(9, 8))
iug_por_loc['mean'].plot(kind='barh', ax=ax, color='#2D5F5F')
ax.set_xlabel('IUG promedio (0-5)')
ax.set_title('IUG promedio por localidad · muestra sintética 200 inmuebles')
ax.axvline(inmuebles.iug.mean(), color='#FFD54F', linestyle='--', label=f'promedio global {inmuebles.iug.mean():.2f}')
ax.legend()
plt.tight_layout()
plt.show()
iug_por_loc

In [ ]:
# Scatter precio vs IUG: si fueran lo mismo veríamos una recta perfecta
fig, ax = plt.subplots(figsize=(9, 6))
sc = ax.scatter(inmuebles.iug, inmuebles.precio/1e6, c=inmuebles.estrato, cmap='viridis', alpha=0.7)
ax.set_xlabel('IUG'); ax.set_ylabel('Precio (millones COP)')
ax.set_title('IUG vs precio · color = estrato')
plt.colorbar(sc, label='estrato')
plt.tight_layout(); plt.show()

## Siguiente paso

- Notebook `02_validacion_hedonica.ipynb` (en progreso): ajuste del modelo hedónico Lasso/Elastic Net contra el corpus completo, bootstrap, ΔAIC y prueba de robustez.
- Notebook `03_busqueda_natural.ipynb` (en progreso): cómo opera el intent parser + spatial search del asistente.

Para reproducir sobre el **corpus completo** (~16.500 inmuebles reales) levanta el backend con `docker compose -f docker-compose.public.yml up -d` y consume `GET /api/inmueble/estadisticas/generales`.